# SongForge-DL — Remote Colab Runner

Runs the project on Colab GPU + Google Drive. Nothing heavy touches the local workstation.

**Run the cells in order, top to bottom.** Each cell is a milestone gate and stops loudly if it fails.

| Cell | Stage | Gate |
| --- | --- | --- |
| 1 | Config | — |
| 2 | Mount Drive + GPU check | GPU present |
| 3 | Get the code onto Colab | project dir found |
| 4 | Install + M00 + M01 | `pytest -q` passes, registry valid |
| 5 | Dataset (needs terms acceptance) | audio present in Drive |
| 6 | M03 codec acceptance | loss falls, RVQ not collapsed |
| 7 | Persist the experiment log to Drive | log copied |

## Before you start

`SETUP_MODE = "zip"` is the default and needs **no GitHub account**.
Upload `songforge-colab.zip` to your Drive at `MyDrive/songforge-dl/` first.

## 1. Config — edit this cell, then run everything below

In [ ]:
# ---- how the code gets onto Colab -------------------------------------
# "zip" = upload songforge-colab.zip to Drive (no GitHub needed)  <-- default
# "git" = clone from GitHub (only if the repo actually exists)
SETUP_MODE = "zip"

DRIVE_ROOT = "/content/drive/MyDrive/songforge-dl"
ZIP_PATH   = f"{DRIVE_ROOT}/songforge-colab.zip"   # upload the archive here
WORK_DIR   = "/content/songforge"                  # local Colab disk: fast. Drive is slow.

# ---- only used when SETUP_MODE == "git" --------------------------------
REPO_URL     = "https://github.com/auth889-ai/ml-sing.git"
BRANCH       = "main"
GITHUB_TOKEN = ""   # fine-grained token if the repo is private; clear cell output afterwards

PROJECT_SUBDIR = "songforge-dl-starter"
SONGFORGE_DATA = f"{DRIVE_ROOT}/data"

# ---- dataset -----------------------------------------------------------
ACCEPT_NONCOMMERCIAL_DATASET_TERMS = False   # set True only after reading the upstream licence
DATASET_ID = "babyslakh"                     # babyslakh, slakh2100, lakh_midi, gtsinger, mtg_jamendo, nsynth

# ---- M03 acceptance ----------------------------------------------------
M03_AUDIO_GLOB = f"{SONGFORGE_DATA}/raw/babyslakh/**/*.wav"
M03_OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/codec_m03_acceptance"
M03_STEPS      = 600   # NOT 80. The RVQ codebook needs warm-up or the collapse gate fails.

print("config loaded")

## 2. Mount Drive and confirm the GPU

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

for sub in ["", "/data/raw", "/data/processed", "/data/manifests",
            "/checkpoints", "/logs", "/outputs"]:
    os.makedirs(DRIVE_ROOT + sub, exist_ok=True)
print("Drive ready:", DRIVE_ROOT)

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Runtime > Change runtime type > Hardware accelerator: GPU, then rerun.")

## 3. Get the code onto Colab

This is the cell that used to fail. It now stops with a clear message instead of
letting later cells run from the wrong directory.

In [ ]:
import os
import shutil
import subprocess
import zipfile
from urllib.parse import urlparse


def fail(message):
    raise SystemExit("SETUP FAILED\n" + message)


if SETUP_MODE == "zip":
    if not os.path.exists(ZIP_PATH):
        fail(
            f"{ZIP_PATH} not found.\n\n"
            "Fix: upload songforge-colab.zip from your Mac to Google Drive at\n"
            f"  MyDrive/songforge-dl/\n"
            "The archive is at ~/connect-your-learning/ml-sing/songforge-colab.zip"
        )
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    os.makedirs(WORK_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(WORK_DIR)
    project_dir = os.path.join(WORK_DIR, PROJECT_SUBDIR)

elif SETUP_MODE == "git":
    repo_dir = os.path.join(WORK_DIR, "repo")
    clone_url = REPO_URL
    if GITHUB_TOKEN:
        parsed = urlparse(REPO_URL)
        if parsed.scheme != "https":
            fail("GITHUB_TOKEN auth expects an https GitHub URL")
        clone_url = f"https://x-access-token:{GITHUB_TOKEN}@{parsed.netloc}{parsed.path}"
    if os.path.exists(repo_dir):
        shutil.rmtree(repo_dir)
    os.makedirs(WORK_DIR, exist_ok=True)
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, clone_url, repo_dir], text=True
    )
    if result.returncode != 0:
        fail(
            "git clone failed.\n"
            "The repository does not exist, is private, or the branch is wrong.\n"
            "Fix: set GITHUB_TOKEN in cell 1, or switch to SETUP_MODE = 'zip'."
        )
    project_dir = os.path.join(repo_dir, PROJECT_SUBDIR)

else:
    fail(f"SETUP_MODE must be 'zip' or 'git', got {SETUP_MODE!r}")

if not os.path.isfile(os.path.join(project_dir, "pyproject.toml")):
    fail(f"pyproject.toml not found in {project_dir}. Check PROJECT_SUBDIR.")

os.chdir(project_dir)
os.environ["SONGFORGE_DATA"] = SONGFORGE_DATA

missing = [
    name
    for name in ("scripts/colab_m03_acceptance.py", "scripts/train_codec.py",
                 "configs/codec/codec_m03_tiny.yaml", "src/songforge/models/codec/quantizer.py")
    if not os.path.exists(name)
]
if missing:
    fail("This build is missing M03 files: " + ", ".join(missing) + "\nRe-export songforge-colab.zip.")

print("project dir :", os.getcwd())
print("M03 files   : present")

## 4. Install + M00 (tests) + M01 (dataset registry)

In [ ]:
import subprocess

!pip install -q -e '.[dev,audio]'

print("\n=== M00: test suite ===")
m00 = subprocess.run(["python", "-m", "pytest", "-q"], text=True)

print("\n=== M01: dataset registry ===")
m01 = subprocess.run(["python", "scripts/validate_dataset_registry.py"], text=True)

if m00.returncode != 0:
    raise SystemExit("M00 FAILED: tests did not pass. Do not continue.")
if m01.returncode != 0:
    raise SystemExit("M01 FAILED: dataset registry invalid. Do not continue.")
print("\nM00 PASS and M01 PASS")

## 5. Dataset

Read the upstream licence first. Gated/non-commercial sets only download after you set
`ACCEPT_NONCOMMERCIAL_DATASET_TERMS = True` in cell 1.

In [ ]:
import subprocess

import yaml

with open("configs/data/datasets.yaml", "r", encoding="utf-8") as f:
    registry = yaml.safe_load(f)

spec = registry["datasets"][DATASET_ID]
needs_acceptance = spec["license"].get("requires_user_acceptance") or spec["access"].get("gated")

print(f"Selected : {DATASET_ID} - {spec['name']}")
print(f"License  : {spec['license']['name']}")
print(f"Size     : {spec['access'].get('estimated_size')}")

if needs_acceptance and not ACCEPT_NONCOMMERCIAL_DATASET_TERMS:
    raise SystemExit(
        "This dataset requires you to accept upstream terms.\n"
        "Read the licence, then set ACCEPT_NONCOMMERCIAL_DATASET_TERMS = True in cell 1."
    )

for command in spec["access"].get("colab_commands", []):
    print("RUN:", command)
    subprocess.run(command, shell=True, check=True)

import glob
found = glob.glob(M03_AUDIO_GLOB, recursive=True)
print(f"\naudio files matching M03_AUDIO_GLOB: {len(found)}")
if not found:
    print("No audio yet. M03 needs real audio at:", M03_AUDIO_GLOB)

## 6. M03 final acceptance

Needs a CUDA runtime and real audio under `M03_AUDIO_GLOB`. It runs the tests, trains the
codec, verifies checkpoint resume, and writes every artifact to Drive.

`M03_STEPS = 600` is deliberate. The residual codebook starts collapsed and needs a few
hundred steps to spread out; at 80 steps the run fails its own `rvq_collapse_suspected` gate.

In [ ]:
!python scripts/colab_m03_acceptance.py \
    --config configs/codec/codec_m03_tiny.yaml \
    --audio-glob "$M03_AUDIO_GLOB" \
    --output-dir "$M03_OUTPUT_DIR" \
    --steps $M03_STEPS

!ls -lh "$M03_OUTPUT_DIR"

## 7. Persist the experiment log to Drive

In [ ]:
import shutil
import json
import os

log_src = "docs/EXPERIMENT_LOG.md"
log_dst = f"{DRIVE_ROOT}/EXPERIMENT_LOG.md"
if os.path.exists(log_src):
    shutil.copy(log_src, log_dst)
    print("copied ->", log_dst)
    print(open(log_src, encoding="utf-8").read()[:4000])
else:
    print("no experiment log yet; run cell 6 first")

report = os.path.join(M03_OUTPUT_DIR, "m03_acceptance_report.json")
if os.path.exists(report):
    acceptance = json.load(open(report))["acceptance"]
    print("\nacceptance_pass       :", acceptance["acceptance_pass"])
    print("loss", acceptance["first_loss"], "->", acceptance["final_loss"])
    print("rvq_collapse_suspected:", acceptance["rvq_collapse_suspected"])